# 3. Phát hiện ngoại biên & gom cụm mật độ với DBSCAN

**Mục tiêu:** Dùng **k-distance graph** để ước lượng bán kính `eps`, quét lưới `(eps, min_samples)`, ghi nhận số cụm cùng tỷ lệ nhiễu (`-1`), và xuất nhãn `dbscan` vào `clustering/results/labels.csv`.

## Lý thuyết cốt lõi
**DBSCAN** (*Density-Based Spatial Clustering of Applications with Noise*) gom các điểm nằm sát nhau theo mật độ dựa trên 2 siêu tham số:
- **`eps` ($\varepsilon$):** Bán kính vùng lân cận xung quanh một điểm.
- **`min_samples`:** Số điểm tối thiểu trong bán kính $\varepsilon$ (tính cả chính nó) để tạo thành vùng đặc.

DBSCAN phân mọi điểm thành 3 nhóm:
1. **Core point (Điểm lõi):** Có ít nhất `min_samples` điểm nằm trong bán kính $\varepsilon$.
2. **Border point (Điểm biên):** Không đủ `min_samples` trong bán kính $\varepsilon$, nhưng nằm trong vùng $\varepsilon$ của một Core point.
3. **Noise (`-1` - Điểm nhiễu / ngoại biên):** Không phải Core cũng không nằm gần Core nào.

## Quyết định áp dụng cho Đề tài 03
- **Mục tiêu chính là phát hiện ngoại biên:** Theo đề bài, DBSCAN dùng để nhận diện các khách hàng bất thường (mua sỉ cực lớn, hành vi dị biệt), nên **không ép** kết quả phải ra đúng 4 cụm như K-Means.
- **Đánh giá chỉ số nội bộ (`ignore_noise=True`):** Nhãn `-1` là tập điểm ngoại biên nằm rải rác chứ không phải một cụm gắn kết, vì vậy các chỉ số Silhouette / Davies-Bouldin / Calinski-Harabasz chỉ tính trên các điểm thuộc cụm hợp lệ (`!= -1`), đồng thời theo dõi riêng cột `noise_ratio`.

In [ ]:
import sys
from pathlib import Path

# Thêm thư mục gốc của repo vào sys.path để import được clustering/models.py
# dù bạn mở notebook từ thư mục gốc hay từ thư mục clustering/
ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "clustering").exists() and (ROOT_DIR.parent / "clustering").exists():
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

from clustering.models import (
    fit_dbscan,
    load_scaled_data,
    run_grid_search,
    save_cluster_labels,
    save_tuning_results,
)

customer_ids, X_df, is_mock = load_scaled_data()
print(f"Kích thước dữ liệu đưa vào mô hình: {X_df.shape}")

## Bước 1: Vẽ k-distance Graph để xác định dải `eps`
- **Cách hoạt động:** Với mỗi điểm, tính khoảng cách tới người hàng xóm gần thứ $k$ (`k = min_samples`) rồi sắp xếp tăng dần.
- **Cách chọn `eps`:** Quan sát điểm "khủy tay" nơi đường cong bắt đầu dựng đứng hẳn lên — dưới ngưỡng này là các điểm nằm trong vùng mật độ dày (thuộc cụm), phía trên là các điểm thưa thớt (ngoại biên).
- **Quy tắc thử `min_samples`:** Với dữ liệu có $d$ đặc trưng, thường thử quanh $d + 1$ đến $2d$ (kèm giá trị nhỏ hơn như `5` để so sánh độ nhạy).

In [ ]:
n_features = X_df.shape[1]
MIN_SAMPLES_GRID = sorted({5, n_features + 1, 2 * n_features})

plt.figure(figsize=(10, 5))
k_distances_map = {}

for k in MIN_SAMPLES_GRID:
    nn = NearestNeighbors(n_neighbors=k).fit(X_df)
    distances, _ = nn.kneighbors(X_df)
    # Cột cuối cùng [:, -1] là khoảng cách tới hàng xóm thứ k
    k_dists = np.sort(distances[:, -1])
    k_distances_map[k] = k_dists
    plt.plot(k_dists, label=f"k = min_samples = {k}")

plt.title("k-distance Graph (Tìm điểm gãy chọn eps)")
plt.xlabel("Điểm dữ liệu (đã sắp xếp tăng dần theo khoảng cách)")
plt.ylabel("Khoảng cách tới hàng xóm thứ k")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## Bước 2: Grid Search `eps` và `min_samples`
- Dải `EPS_GRID` dưới đây được ước lượng tự động quanh vùng phân vị cao của `k-distance` (nơi đường cong bắt đầu uốn lên) để bạn chạy thử nhanh trên mọi tập dữ liệu. Sau khi xem đồ thị trên dữ liệu thật của TV1, bạn có thể thay bằng danh sách số cụ thể quanh điểm gãy.
- **Lưu ý quan trọng khi lọc kết quả:** Loại bỏ cấu hình chỉ ra 1 cụm (`n_clusters < 2`) hoặc tỷ lệ nhiễu quá cao (`noise_ratio >= 0.30`). Vì nếu `eps` quá nhỏ khiến 35–50% khách hàng bị gán `-1`, chỉ số Silhouette trên phần còn lại sẽ tăng giả tạo nhưng mất ý nghĩa phân khúc.

In [ ]:
# Tạo dải eps gợi ý từ k-distance (có thể thay bằng list số cụ thể, vd: [0.8, 1.0, 1.2, 1.5])
ref_dists = k_distances_map[min(MIN_SAMPLES_GRID)]
EPS_GRID = sorted(
    set(
        float(v)
        for v in np.round(np.quantile(ref_dists, [0.50, 0.70, 0.80, 0.85, 0.90, 0.94, 0.97]), 2)
        if v > 0
    )
)
print("Dải eps thử nghiệm:", EPS_GRID)
print("Dải min_samples thử nghiệm:", MIN_SAMPLES_GRID)

tuning_df = run_grid_search(
    X_df,
    model_builder=lambda eps, min_samples: DBSCAN(eps=eps, min_samples=min_samples),
    param_grid={"eps": EPS_GRID, "min_samples": MIN_SAMPLES_GRID},
    ignore_noise=True,  # Tính Silhouette/DB/CH sau khi tách điểm nhiễu (-1)
)

save_tuning_results(tuning_df, "tuning_dbscan.csv")

# Ưu tiên các cấu hình có >= 2 cụm và tỷ lệ nhiễu < 30% (tránh bỏ quá nhiều điểm làm Silhouette tăng ảo)
valid_df = tuning_df[
    (tuning_df["n_clusters"] >= 2) & (tuning_df["noise_ratio"] < 0.30)
].sort_values(by=["silhouette", "noise_ratio"], ascending=[False, True])

if valid_df.empty:
    valid_df = tuning_df[
        (tuning_df["n_clusters"] >= 2) & (tuning_df["noise_ratio"] < 0.50)
    ].sort_values(by=["silhouette", "noise_ratio"], ascending=[False, True])

print("Top cấu hình hợp lệ:")
valid_df.head(10)

## Bước 3: Biểu đồ độ nhạy của `eps` đến Số cụm và Tỷ lệ nhiễu
Biểu đồ này cho thấy rõ sự đánh đổi: `eps` quá nhỏ khiến quá nhiều khách hàng bị coi là nhiễu (`noise_ratio` cao); `eps` quá lớn lại khiến các cụm dính vào nhau còn 1 cụm và bỏ sót ngoại biên.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ms in MIN_SAMPLES_GRID:
    sub = tuning_df[tuning_df["min_samples"] == ms].sort_values("eps")
    axes[0].plot(sub["eps"], sub["n_clusters"], marker="o", label=f"min_samples={ms}")
    axes[1].plot(sub["eps"], sub["noise_ratio"] * 100, marker="s", label=f"min_samples={ms}")

axes[0].set_title("Số cụm theo eps")
axes[0].set_xlabel("eps")
axes[0].set_ylabel("Số cụm (không tính nhãn -1)")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.5)

axes[1].set_title("Tỷ lệ điểm nhiễu (-1) theo eps")
axes[1].set_xlabel("eps")
axes[1].set_ylabel("Tỷ lệ nhiễu (%)")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## Bước 4: Chốt tham số cuối & Lưu nhãn bàn giao
> **Ghi chú cho báo cáo Chương 4 (cập nhật sau khi chạy trên data thật):**
> - Bộ tham số lựa chọn `BEST_EPS = ...`, `BEST_MIN_SAMPLES = ...` vì tạo ra `...` cụm rõ ràng và tách được khoảng `...%` điểm ngoại biên (nhãn `-1`) cho TV3 phân tích chuyên sâu.

In [ ]:
# >>> CHỈNH THAM SỐ CỤ THỂ SAU KHI XEM BẢNG VÀ BIỂU ĐỒ BÊN TRÊN <<<
# Ví dụ: BEST_EPS = 1.5 ; BEST_MIN_SAMPLES = 10
BEST_EPS = None
BEST_MIN_SAMPLES = None

# Nếu chưa gán tay, tạm chọn cấu hình tốt nhất trong valid_df để khung chạy được trọn vẹn
if BEST_EPS is None or BEST_MIN_SAMPLES is None:
    if not valid_df.empty:
        best_row = valid_df.iloc[0]
        BEST_EPS = float(best_row["eps"])
        BEST_MIN_SAMPLES = int(best_row["min_samples"])
        print(f"[TỰ ĐỘNG CHỌN TẠM] eps={BEST_EPS}, min_samples={BEST_MIN_SAMPLES}")
    else:
        best_row = tuning_df.sort_values("noise_ratio").iloc[0]
        BEST_EPS = float(best_row["eps"])
        BEST_MIN_SAMPLES = int(best_row["min_samples"])
        print(f"[CẢNH BÁO] Chưa có cấu hình >= 2 cụm; tạm dùng eps={BEST_EPS}, min_samples={BEST_MIN_SAMPLES}")

final_dbscan, dbscan_labels = fit_dbscan(
    X_df,
    eps=BEST_EPS,
    min_samples=BEST_MIN_SAMPLES,
)

counts = pd.Series(dbscan_labels, name="count").value_counts().sort_index()
noise_pct = (dbscan_labels == -1).mean() * 100
print(f"Phân bố nhãn DBSCAN (tỷ lệ ngoại biên -1: {noise_pct:.2f}%):")
print(counts)

save_cluster_labels(customer_ids, "dbscan", dbscan_labels)